# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MRazaRashid/FlyRank_Week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import duckdb
import os
from google.colab import userdata

hf_token=userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [2]:
features= con.sql("""
SELECT
  client_hash_id , content_hash_id,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
  AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS avg_position_first_half,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN sessions_organic ELSE 0 END) AS organic_sessions_first_half,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN scroll_events ELSE 0 END) AS scroll_events_first_half
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id , content_hash_id

""").df()
features["avg_position_first_half"] = features["avg_position_first_half"].fillna(-1)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_first_half,avg_position_first_half,clicks_first_half,organic_sessions_first_half,scroll_events_first_half
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,3.659683,0.0,0.0,0.0
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,4974.0,8.155725,9.0,0.0,0.0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,35.0,5.177778,0.0,0.0,0.0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,398.0,4.667061,1.0,0.0,0.0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,37.0,18.079060,0.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

There is no categorical column in my features
# **impressions_first_half**
Total GSC impressions, Mar 1-15

Mising values Filled with 0

# **avg_position_first_half**
Average search ranking position, Mar 1-15


Filled with -1 (not 0 — since avg_position = 0 in this dataset means "no data," not rank zero)

# **clicks_first_half**
Total GSC clicks, Mar 1-15

Mising values Filled with 0

# **organic_sessions_first_half**

Total organic traffic sessions, Mar 1-15

Mising values Filled with 0

# **scroll_events_first_half**
Total scroll-depth events, Mar 1-15

Mising values Filled with 0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
labeled = con.sql("""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
  SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_second_half
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

In [ ]:
labeled["is_declining_label"] = (
    (labeled["impressions_first_half"] >= 50) &
    (labeled["impressions_second_half"] < labeled["impressions_first_half"] * 0.8)
).astype(int)

In [ ]:
full = features.merge(
    labeled[["client_hash_id", "content_hash_id", "impressions_second_half", "is_declining_label"]],
    on=["client_hash_id", "content_hash_id"]
)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.